In [0]:
from pyspark.sql import functions as F

silver_table = "silver_telemetry"
gold_table = "gold_thermal_metrics"
gold_checkpoint = "/Volumes/workspace/default/thermal_data/_checkpoints/gold/"

In [0]:
# 1. Read cleaned stream from Silver
silver_stream = spark.readStream.table(silver_table)

# 2. Windowed Aggregations with Watermark
gold_aggregated_df = (
    silver_stream
    .withWatermark("timestamp", "10 seconds")
    .groupBy(
        F.window("timestamp", "10 seconds")
    )
    .agg(
        F.round(F.avg("cpu_utilization"), 2).alias("avg_cpu_utilization"),
        F.round(F.avg("gpu_utilization"), 2).alias("avg_gpu_utilization"),
        F.round(F.avg("power_draw_watts"), 2).alias("avg_power_watts"),
        F.round(F.max("power_draw_watts"), 2).alias("peak_power_watts"),
        F.round(F.avg("current_fluid_temp"), 2).alias("avg_fluid_temp"),
        F.round(F.max("current_fluid_temp"), 2).alias("max_fluid_temp"),
        F.sum(F.when(F.col("is_thermal_alert") == True, 1).otherwise(0)).alias("alert_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "avg_cpu_utilization",
        "avg_gpu_utilization",
        "avg_power_watts",
        "peak_power_watts",
        "avg_fluid_temp",
        "max_fluid_temp",
        "alert_count"
    )
)

# 3. Write Stream to Delta Gold Table
gold_query = (
    gold_aggregated_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .toTable(gold_table)
)

gold_query.awaitTermination()

In [0]:
display(
    spark.table(gold_table)
    .orderBy(F.col("window_start").desc())
)

In [0]:
from pyspark.sql import functions as F

gold_table = "workspace.default.gold_thermal_metrics"

gold_batch_df = (
    spark.table("workspace.default.silver_telemetry")
    .groupBy(F.window("timestamp", "10 seconds"))
    .agg(
        F.round(F.avg("cpu_utilization"), 2).alias("avg_cpu_utilization"),
        F.round(F.avg("gpu_utilization"), 2).alias("avg_gpu_utilization"),
        F.round(F.avg("power_draw_watts"), 2).alias("avg_power_watts"),
        F.round(F.max("power_draw_watts"), 2).alias("peak_power_watts"),
        F.round(F.avg("current_fluid_temp"), 2).alias("avg_fluid_temp"),
        F.round(F.max("current_fluid_temp"), 2).alias("max_fluid_temp"),
        F.sum(F.when(F.col("is_thermal_alert") == True, 1).otherwise(0)).alias("alert_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "avg_cpu_utilization",
        "avg_gpu_utilization",
        "avg_power_watts",
        "peak_power_watts",
        "avg_fluid_temp",
        "max_fluid_temp",
        "alert_count"
    )
)

gold_batch_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)